# ⚖️ Binary Classification – ERD/ERS Dataset (Modified Preprocessing)
**Task:** Classify between **any 2 of the 9 EEG scenarios** (user-selectable)  
**Output folder:** `binary_erd_ers_dataset_outputs/`

> Uses `erd_ers_band_power.csv`. Pivoted to wide format, then stratified split.  
> Simply change `SCENARIO_A` and `SCENARIO_B` to any pair from 1–9.

### Robust Preprocessing Pipeline
EEG data is highly subject-dependent. This notebook applies:
1. **Outlier clipping** – ERD/ERS values clipped at ±500% (physiologically plausible range)
2. **Within-subject z-score normalization** – removes inter-subject baseline differences before pivoting
3. **Robust scaling (IQR-based)** – `RobustScaler` instead of `StandardScaler` to handle residual outliers
4. **Variance-based feature filtering** – removes near-zero-variance features post-pivot
5. **Subject-group-aware CV** – `GroupKFold` ensures subjects never leak across folds

### 9 Available Scenarios
| # | Scenario |
|---|----------|
| 1 | Lifting the left hand |
| 2 | Lifting the right hand |
| 3 | Lifting the left leg |
| 4 | Lifting the right leg |
| 5 | Opening the mouth |
| 6 | Nodding the head |
| 7 | Shaking head |
| 8 | Desiring to drink water |
| 9 | Desiring to use the bathroom |


## 0. Setup


In [1]:
import os, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import (StratifiedKFold, cross_val_score,
                                     train_test_split, GroupKFold)
from sklearn.preprocessing import RobustScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import VarianceThreshold
from sklearn.ensemble import (RandomForestClassifier, GradientBoostingClassifier,
                               AdaBoostClassifier, StackingClassifier)
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.metrics import (classification_report, confusion_matrix,
                              accuracy_score, f1_score, roc_auc_score,
                              RocCurveDisplay)
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

plt.rcParams.update({'figure.dpi': 120, 'font.size': 10})

SCENARIO_LABELS = {
    1: 'Lifting the left hand',
    2: 'Lifting the right hand',
    3: 'Lifting the left leg',
    4: 'Lifting the right leg',
    5: 'Opening the mouth',
    6: 'Nodding the head',
    7: 'Shaking head',
    8: 'Desiring to drink water',
    9: 'Desiring to use the bathroom',
}
print('Setup complete.')

Setup complete.


In [2]:
# ════════════════════════════════════════════════════════════════
#  ▶ CHOOSE YOUR TWO SCENARIOS HERE
# ════════════════════════════════════════════════════════════════
SCENARIO_A = 5    # ← Replace with any integer 1-9
SCENARIO_B = 7    # ← Replace with any integer 1-9 (must differ from A)
# ════════════════════════════════════════════════════════════════

assert SCENARIO_A != SCENARIO_B, 'Scenarios must be different!'
assert SCENARIO_A in range(1,10) and SCENARIO_B in range(1,10), 'Must be 1-9'

label_A = SCENARIO_LABELS[SCENARIO_A]
label_B = SCENARIO_LABELS[SCENARIO_B]
print(f'Binary Classification:')
print(f'  Class 0 → S{SCENARIO_A}: {label_A}')
print(f'  Class 1 → S{SCENARIO_B}: {label_B}')

Binary Classification:
  Class 0 → S5: Opening the mouth
  Class 1 → S7: Shaking head


## 1. Load & Robust Preprocessing

EEG data has extreme inter-subject variability. The pipeline:
- **Step 1 – Outlier clipping**: ERD/ERS values outside `[-100, 500]%` (physiologically implausible) are clipped
- **Step 2 – Within-subject z-score**: Each subject's values are z-scored independently **before** pivoting, so the model sees relative patterns, not absolute amplitudes
- **Step 3 – Pivot to wide**: One row per subject, all channel × band × metric combinations as columns
- **Step 4 – Median imputation**: Fill any remaining NaN after pivot
- **Step 5 – Variance filter**: Drop near-zero-variance features (`threshold=0.01`)


In [ ]:
PATH = "erd_ers_band_power.csv"
df = pd.read_csv(PATH)

# Normalize column names to avoid KeyError from hidden spaces/case drift.
df.columns = [c.strip() for c in df.columns]

# Handle common alias variants across exports.
rename_map = {
    'Subject': 'subject',
    'Subject ID': 'subject',
    'Subject_ID': 'subject',
    'Scenario': 'scenario',
    'Channel': 'channel',
    'Task': 'task',
    'Subband': 'subband',
    'Time': 'time',
    'ERD_ERS_PCT': 'erd_ers_pct',
}
df = df.rename(columns={k: v for k, v in rename_map.items() if k in df.columns})

required_cols = ['subject', 'scenario', 'task', 'channel', 'subband', 'erd_ers_pct']
missing_cols = [c for c in required_cols if c not in df.columns]
if missing_cols:
    raise KeyError(f"Missing required columns: {missing_cols}. Found: {list(df.columns)}")

# Parse scenario number
df['scenario_num'] = df['scenario'].astype(str).str.extract(r'(\d+)').astype(int)

# Remove rows with blank/whitespace channel
df = df[df['channel'].astype(str).str.strip() != ''].copy()

# Filter to selected two scenarios
df = df[df['scenario_num'].isin([SCENARIO_A, SCENARIO_B])].copy()
print(f'Filtered rows : {df.shape}')
print(f'Subjects      : {df["subject"].nunique()}')
print(f'Scenarios     : {df["scenario_num"].unique()}')

# ─────────────────────────────────────────────────────────────────────
# STEP 1: Clip extreme ERD/ERS outliers
# ERD/ERS% = (task - baseline) / baseline * 100
# Values > 500% or < -100% are physiologically implausible noise.
# ─────────────────────────────────────────────────────────────────────
CLIP_LOW, CLIP_HIGH = -100.0, 500.0
df['erd_ers_pct'] = df['erd_ers_pct'].clip(CLIP_LOW, CLIP_HIGH)
df['baseline_power (µ)'] = df['baseline_power (µ)'].clip(lower=0)   # power is non-negative
df['task_power (µ)']     = df['task_power (µ)'].clip(lower=0)
print(f'\nAfter clipping ERD/ERS to [{CLIP_LOW}, {CLIP_HIGH}]:')
print(df['erd_ers_pct'].describe().to_string())


Filtered rows : (3168, 12)
Subjects      : 176
Scenarios     : [5 7]

After clipping ERD/ERS to [-100.0, 500.0]:
count    3168.000000
mean      160.141502
std       221.893115
min      -100.000000
25%       -23.166201
50%        74.672543
75%       420.430307
max       500.000000


In [4]:
# ─────────────────────────────────────────────────────────────────────
# STEP 2: Within-subject z-score normalization
# EEG amplitude is subject-dependent (skull thickness, electrode impedance,
# amplifier gain, etc.). Normalizing within each subject removes this
# nuisance and forces the model to learn *relative* spectral patterns.
# ─────────────────────────────────────────────────────────────────────
numeric_cols = ['baseline_power (µ)', 'task_power (µ)', 'erd_ers_pct']

def zscore_within_subject(group):
    for col in numeric_cols:
        mu  = group[col].mean()
        std = group[col].std()
        # guard against flat subjects (std ≈ 0)
        group[col] = (group[col] - mu) / (std + 1e-8)
    return group

df = df.groupby('subject', group_keys=False).apply(zscore_within_subject)
print('Within-subject z-score applied.')
print(df[numeric_cols].describe().round(4).to_string())


Within-subject z-score applied.
       baseline_power (µ)  task_power (µ)  erd_ers_pct
count           3168.0000       3168.0000    3168.0000
mean              -0.0000          0.0000       0.0000
std                0.9662          0.9689       0.9720
min               -1.8858         -1.9985      -4.0069
25%               -0.6853         -0.6426      -0.7736
50%               -0.3434         -0.3909      -0.2256
75%                0.4479          0.3547       0.7495
max                3.8141          3.8226       3.5535


In [ ]:
# ─────────────────────────────────────────────────────────────────────
# STEP 3: Pivot to wide format (one row per subject)
# ─────────────────────────────────────────────────────────────────────
# Recover keys if they ended up in the index due to prior notebook state.
if 'subject' not in df.columns and getattr(df.index, 'name', None) == 'subject':
    df = df.reset_index()
if 'scenario_num' not in df.columns and getattr(df.index, 'name', None) == 'scenario_num':
    df = df.reset_index()

if 'time' not in df.columns:
    df['time'] = 't1'

df['col_key'] = (df['task'].astype(str) + '_' + df['channel'].astype(str) + '_' +
                 df['subband'].astype(str) + '_' + df['time'].fillna('t1').astype(str))

pivot = df.pivot_table(
    index=['subject', 'scenario_num'],
    columns='col_key',
    values=['baseline_power (µ)', 'task_power (µ)', 'erd_ers_pct'],
    aggfunc='mean'
).reset_index()

pivot.columns = ['_'.join(filter(None, map(str, c))).strip('_')
                 if isinstance(c, tuple) else c
                 for c in pivot.columns]

id_cols      = ['subject', 'scenario_num']
feature_cols = [c for c in pivot.columns if c not in id_cols]

X_raw = pivot[feature_cols].values.astype(float)
y     = (pivot['scenario_num'].values == SCENARIO_B).astype(int)
groups = pivot['subject'].values   # keep for group-aware CV

# ─────────────────────────────────────────────────────────────────────
# STEP 4: Median imputation (handles NaN after pivot)
# ─────────────────────────────────────────────────────────────────────
imputer = SimpleImputer(strategy='median')
X_imp = imputer.fit_transform(X_raw)

# ─────────────────────────────────────────────────────────────────────
# STEP 5: Variance-based feature filtering
# After per-subject z-scoring many columns may have near-zero variance
# (e.g. all subjects showing exactly 0 for a given band-channel combo).
# Removing them reduces noise and speeds up training.
# ─────────────────────────────────────────────────────────────────────
vt = VarianceThreshold(threshold=0.01)
X = vt.fit_transform(X_imp)
feature_cols = [f for f, keep in zip(feature_cols, vt.get_support()) if keep]

OUTPUT_DIR = f'binary_erd_ers_preprocessingModified_S{SCENARIO_A}_vs_S{SCENARIO_B}_outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f'Pivoted shape  : {pivot.shape}')
print(f'After impute   : {X_imp.shape[1]} features')
print(f'After var filter: {X.shape[1]} features retained')
print(f'Class balance  : 0={(y==0).sum()}  1={(y==1).sum()}')


KeyError: 'subject'

## 2. Train / Test Split

> **Subject-group-aware split**: subjects are kept whole in either train or test — no subject can appear in both. This prevents data leakage and gives a realistic estimate of generalisation to new subjects.


In [ ]:
# Group-aware split: keep all rows of each subject in one partition only.
# This avoids the optimistic bias that comes from having the same subject
# in both train and test (which the original stratified split allowed).
from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train, X_test = X[train_idx], X[test_idx]
y_train, y_test = y[train_idx], y[test_idx]
groups_train    = groups[train_idx]

print(f'Train : {X_train.shape}  classes: 0={(y_train==0).sum()} 1={(y_train==1).sum()}')
print(f'Test  : {X_test.shape}   classes: 0={(y_test==0).sum()} 1={(y_test==1).sum()}')
print(f'Train subjects: {len(np.unique(groups_train))} | Test subjects: {len(np.unique(groups[test_idx]))}')
print(f'Subject overlap: {set(np.unique(groups_train)) & set(np.unique(groups[test_idx]))} (should be empty)')


In [ ]:
# ── Model Pipelines  (RobustScaler replaces StandardScaler) ─────────────────
# RobustScaler uses median and IQR instead of mean and std, making it
# far less sensitive to the residual outliers still present in EEG data.

models = {
    'Random Forest': Pipeline([
        ('sc',  RobustScaler()),
        ('clf', RandomForestClassifier(n_estimators=300, max_depth=15,
                                       random_state=42, n_jobs=-1))
    ]),
    'Logistic Regression': Pipeline([
        ('sc',  RobustScaler()),
        ('clf', LogisticRegression(max_iter=2000, C=1.0, solver='lbfgs',
                                   random_state=42))
    ]),
    'SVM (RBF)': Pipeline([
        ('sc',  RobustScaler()),
        ('clf', SVC(kernel='rbf', C=10, gamma='scale',
                    probability=True, random_state=42))
    ]),
    'KNN': Pipeline([
        ('sc',  RobustScaler()),
        ('clf', KNeighborsClassifier(n_neighbors=7, n_jobs=-1))
    ]),
    'XGBoost': Pipeline([
        ('sc',  RobustScaler()),
        ('clf', XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.1,
                              subsample=0.8, colsample_bytree=0.8,
                              eval_metric='logloss', random_state=42,
                              n_jobs=-1, tree_method='hist'))
    ]),
    'LightGBM': Pipeline([
        ('sc',  RobustScaler()),
        ('clf', LGBMClassifier(n_estimators=300, num_leaves=63, learning_rate=0.1,
                               random_state=42, n_jobs=-1, verbose=-1))
    ]),
    'AdaBoost': Pipeline([
        ('sc',  RobustScaler()),
        ('clf', AdaBoostClassifier(n_estimators=150, learning_rate=0.5,
                                   random_state=42))
    ]),
    'Gradient Boosting': Pipeline([
        ('sc',  RobustScaler()),
        ('clf', GradientBoostingClassifier(n_estimators=200, max_depth=5,
                                           learning_rate=0.1, random_state=42))
    ]),
}

stk_est = [
    ('rf',   RandomForestClassifier(n_estimators=150, random_state=42, n_jobs=-1)),
    ('xgb',  XGBClassifier(n_estimators=150, tree_method='hist',
                            eval_metric='logloss', random_state=42, n_jobs=-1)),
    ('lgbm', LGBMClassifier(n_estimators=150, random_state=42, verbose=-1)),
    ('svm',  SVC(probability=True, random_state=42)),
]
models['Stacking (RF+XGB+LGBM+SVM)'] = Pipeline([
    ('sc',  RobustScaler()),
    ('clf', StackingClassifier(
        estimators=stk_est,
        final_estimator=LogisticRegression(max_iter=2000, random_state=42),
        cv=5, n_jobs=-1
    ))
])
print(f'Models ready: {list(models.keys())}')


In [ ]:
# ── Cross-Validation (GroupKFold – subject-aware) ────────────────────────────
# GroupKFold guarantees that the same subject never appears in both the
# train fold and the validation fold. This is critical for EEG: a naive
# StratifiedKFold would randomly split rows, putting different trials of
# the same subject in both partitions, inflating CV scores.
N_SPLITS = 5   # use 5 instead of 10 to respect subject count limits
gkf = GroupKFold(n_splits=N_SPLITS)
cv_results = {}
print(f'{N_SPLITS}-Fold GroupKFold CV (subject-aware)...\n')
for name, pipe in models.items():
    if 'Stacking' in name:
        print(f'  {name:40s}: [skipped in CV]')
        continue
    scores = cross_val_score(pipe, X_train, y_train,
                             cv=gkf.split(X_train, y_train, groups_train),
                             scoring='roc_auc', n_jobs=-1)
    cv_results[name] = scores
    print(f'  {name:40s}: AUC={scores.mean():.4f} ± {scores.std():.4f}')


In [ ]:
# ── Train & Test Evaluation ───────────────────────────────────────────────────
test_results = {}
for name, pipe in models.items():
    print(f'  Fitting {name} ...', end='  ', flush=True)
    pipe.fit(X_train, y_train)
    y_pred  = pipe.predict(X_test)
    y_proba = pipe.predict_proba(X_test)[:, 1]
    acc  = accuracy_score(y_test, y_pred)
    f1   = f1_score(y_test, y_pred)
    auc  = roc_auc_score(y_test, y_proba)
    test_results[name] = {
        'accuracy': acc, 'f1': f1, 'auc': auc,
        'y_pred': y_pred, 'y_proba': y_proba
    }
    print(f'Acc={acc:.4f}  F1={f1:.4f}  AUC={auc:.4f}')


In [ ]:
# ── FIG 1 – CV AUC Comparison ─────────────────────────────────────────────────
cv_names = list(cv_results.keys())
cv_means = [cv_results[n].mean() for n in cv_names]
cv_stds  = [cv_results[n].std()  for n in cv_names]
fig, ax = plt.subplots(figsize=(11, 4))
bars = ax.barh(cv_names, cv_means, xerr=cv_stds, height=0.5,
               color=plt.cm.Set2(np.linspace(0,1,len(cv_names))),
               alpha=0.85, error_kw=dict(ecolor='gray', capsize=4))
for bar, v in zip(bars, cv_means):
    ax.text(v+0.003, bar.get_y()+bar.get_height()/2, f'{v:.4f}', va='center', fontsize=9)
ax.axvline(0.5, color='red', ls='--', alpha=0.5, label='Chance (AUC=0.5)')
ax.set_xlabel('ROC-AUC'); ax.set_xlim(0, 1.1)
ax.set_title(f'{N_SPLITS}-Fold GroupKFold CV ROC-AUC – Binary: S{SCENARIO_A} vs S{SCENARIO_B}',
             fontsize=12, fontweight='bold')
ax.legend(); ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/01_cv_auc.png', bbox_inches='tight')
plt.show()


In [ ]:
# ── FIG 2 – Test performance (Acc / F1 / AUC) ────────────────────────────────
nt   = list(test_results.keys())
accs = [test_results[n]['accuracy'] for n in nt]
f1s  = [test_results[n]['f1']       for n in nt]
aucs = [test_results[n]['auc']      for n in nt]
x = np.arange(len(nt)); w = 0.25

fig, ax = plt.subplots(figsize=(15, 5))
b1 = ax.bar(x-w,   accs, w, label='Accuracy',  color='steelblue',  alpha=0.85)
b2 = ax.bar(x,     f1s,  w, label='F1-Score',  color='darkorange', alpha=0.85)
b3 = ax.bar(x+w,   aucs, w, label='ROC-AUC',   color='mediumseagreen', alpha=0.85)
for b in list(b1)+list(b2)+list(b3):
    ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.01,
            f'{b.get_height():.3f}', ha='center', fontsize=7)
ax.axhline(0.5, color='red', ls='--', alpha=0.4, label='Chance')
ax.set_xticks(x); ax.set_xticklabels(nt, rotation=25, ha='right', fontsize=9)
ax.set_ylabel('Score'); ax.set_ylim(0, 1.2)
ax.set_title(f'Test Performance – S{SCENARIO_A} vs S{SCENARIO_B}',
             fontsize=11, fontweight='bold')
ax.legend(); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/02_test_performance.png', bbox_inches='tight')
plt.show()


In [ ]:
# ── FIG 3 – ROC Curves (all models) ──────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 7))
colors = plt.cm.tab10(np.linspace(0, 1, len(test_results)))
for (name, res), color in zip(test_results.items(), colors):
    RocCurveDisplay.from_predictions(
        y_test, res['y_proba'],
        name=f"{name} (AUC={res['auc']:.3f})",
        ax=ax, color=color, lw=1.5)
ax.plot([0,1],[0,1],'k--', alpha=0.4, label='Chance')
ax.set_title(f'ROC Curves – S{SCENARIO_A} vs S{SCENARIO_B}',
             fontsize=12, fontweight='bold')
ax.legend(loc='lower right', fontsize=7)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/03_roc_curves.png', bbox_inches='tight')
plt.show()


In [ ]:
# ── FIG 4 – Best model confusion matrix ──────────────────────────────────────
best_name = max(test_results, key=lambda n: test_results[n]['auc'])
print(f'Best: {best_name}  AUC={test_results[best_name]["auc"]:.4f}')
cm = confusion_matrix(y_test, test_results[best_name]['y_pred'])
labels_bin = [f'S{SCENARIO_A}: {label_A[:15]}', f'S{SCENARIO_B}: {label_B[:15]}']
fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=labels_bin, yticklabels=labels_bin, linewidths=0.5,
            annot_kws={'size': 14})
ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
ax.set_title(f'Confusion Matrix – {best_name}\nS{SCENARIO_A} vs S{SCENARIO_B}',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/04_confusion_matrix_best.png', bbox_inches='tight')
plt.show()


In [ ]:
# ── FIG 5 – Classification report ────────────────────────────────────────────
print(classification_report(y_test, test_results[best_name]['y_pred'],
      target_names=labels_bin))


In [ ]:
# ── FIG 6 – RF Feature Importance ────────────────────────────────────────────
rf_imp = models['Random Forest'].named_steps['clf'].feature_importances_
fi = (pd.DataFrame({'feature': feature_cols, 'importance': rf_imp})
      .sort_values('importance', ascending=False).head(20))
fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(fi['feature'], fi['importance'],
        color=plt.cm.YlOrRd(np.linspace(0.4, 0.9, 20))[::-1])
ax.set_xlabel('Importance')
ax.set_title(f'Top 20 Features – RF  (S{SCENARIO_A} vs S{SCENARIO_B})',
             fontsize=12, fontweight='bold')
ax.invert_yaxis(); ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/05_rf_feature_importance.png', bbox_inches='tight')
plt.show()


In [ ]:
# ── FIG 7 – All confusion matrices grid ──────────────────────────────────────
nm = len(test_results); ncols = 4; nrows = (nm+ncols-1)//ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(4*ncols, 4*nrows))
fig.suptitle(f'All CMs – S{SCENARIO_A} vs S{SCENARIO_B}', fontsize=13, fontweight='bold')
cmaps_b = ['Blues','Greens','Oranges','Reds','Purples','YlOrBr','GnBu','RdPu','BuPu']
for idx, (name, res) in enumerate(test_results.items()):
    ax  = axes.flat[idx]
    cmi = confusion_matrix(y_test, res['y_pred'])
    sns.heatmap(cmi, annot=True, fmt='d', cmap=cmaps_b[idx%len(cmaps_b)],
                ax=ax, xticklabels=[f'S{SCENARIO_A}',f'S{SCENARIO_B}'],
                yticklabels=[f'S{SCENARIO_A}',f'S{SCENARIO_B}'],
                linewidths=0.5, cbar=False)
    ax.set_title(f"{name}\nAUC={res['auc']:.3f}", fontsize=8, fontweight='bold')
    ax.tick_params(labelsize=8)
for idx in range(nm, nrows*ncols): axes.flat[idx].axis('off')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/06_all_confusion_matrices.png', bbox_inches='tight')
plt.show()


In [ ]:
# ── Save Results ──────────────────────────────────────────────────────────────
summary = pd.DataFrame([{
    'Model':    n,
    'Accuracy': r['accuracy'],
    'F1_Score': r['f1'],
    'ROC_AUC':  r['auc'],
    'CV_AUC_Mean': cv_results[n].mean() if n in cv_results else None,
    'CV_AUC_Std':  cv_results[n].std()  if n in cv_results else None,
} for n, r in test_results.items()]).sort_values('ROC_AUC', ascending=False)

out_file = f'{OUTPUT_DIR}/binary_S{SCENARIO_A}_vs_S{SCENARIO_B}_results.csv'
summary.to_csv(out_file, index=False)
print(summary.to_string(index=False))
print(f'\n✅ Saved to {out_file}')
